# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema accessible via the following URL:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json


In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}. Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, their fields, and `@id` values.

In [ ]:
# List all record sets and their fields, referencing by @id
print("Record Sets available:")
record_sets = [rs for rs in dataset.record_sets]
for rs in record_sets:
    print(f"- Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}) | type: {field.data_type}")
    print()

# Show the first record set and a sample record (if available)
if record_sets:
    rs = record_sets[0]
    print(f"Sample record from record set {rs.name} (@id: {rs.id}):")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs.id)):
            print(rec)
            if i >= 1:  # Print only the first two for preview
                break
    except Exception as e:
        print(f"Could not print sample records: {e}")

## 3. Data Extraction
Load data from each record set into pandas DataFrames for analysis using `@id` references.

In [ ]:
# Extract all record sets using their @id
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded DataFrame for {rs_id}: {df.shape[0]} rows, {df.shape[1]} columns")
    except Exception as e:
        print(f"Skipping {rs_id} due to error: {e}")

# Display columns and a preview for the first record set (if any)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nColumns in {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply exploratory data processing: filter records, normalize numerical columns, group by key attributes.

**All column and field references are made by their `@id`.**

In [ ]:
# Example: EDA on the main record set
main_rs_id = record_set_ids[0]
df_main = dataframes[main_rs_id]

# Show all available column @ids
print("Columns in DataFrame:")
for col in df_main.columns:
    print(f"- {col}")

# Select a numeric field by @id (if available)
import numpy as np

# Let's heuristically find an integer/float column to use as a numeric field
numeric_field_id = None
for col in df_main.columns:
    if np.issubdtype(df_main[col].dropna().apply(type).mode().values[0], (int,float)) or df_main[col].dtype in [np.int64, np.float64]:
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Try as fallback to columns with names suggesting numeric values
    for col in df_main.columns:
        if any(s in col.lower() for s in ['age', 'interval', 'count', 'number', 'years', 'months']):
            numeric_field_id = col
            break

if numeric_field_id is None:
    print("No numeric field found for analysis.")
else:
    print(f"Using numeric field: {numeric_field_id}")
    # Try converting to numeric (if it's not already)
    df_main[numeric_field_id] = pd.to_numeric(df_main[numeric_field_id], errors='coerce')
    # Set a threshold (median) and filter
    threshold = df_main[numeric_field_id].median()
    filtered_df = df_main[df_main[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric_field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (e.g., 'sex' or 'status' if available)
    group_field_id = None
    for col in df_main.columns:
        if col != numeric_field_id and df_main[col].nunique() > 1 and df_main[col].nunique() < df_main.shape[0]//2:
            group_field_id = col
            break
    if group_field_id is not None:
        print(f"Grouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'count'])
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field, and show group-wise statistics if grouping was possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field (if available)
if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df_main[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_main)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
We have successfully explored and processed the FAIR² colorectal cancer survivors dataset using Croissant and `mlcroissant`.

- **Record sets and fields were referenced by their `@id`, ensuring reproducibility and clarity.**
- **We loaded all dataset records dynamically using the Croissant metadata.**
- **A basic exploratory data analysis and visualization illustrated key data attributes.**

Refer to the full dataset description and documentation for responsible and appropriate use.
